# 01 — Data Ingestion
Loads and validates the 6 raw source files for the Bluestock MF platform. This mirrors the logic in `scripts/etl_pipeline.py` — see that script for the production/CLI version of this same process.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

files = {
    "fund_master": "01_fund_master.csv",
    "nav_history": "02_nav_history.csv",
    "scheme_performance": "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings": "09_portfolio_holdings.csv",
    "benchmark_indices": "10_benchmark_indices.csv",
}

datasets = {}
for name, fname in files.items():
    path = RAW_DIR / fname
    if not path.exists():
        raise FileNotFoundError(f"Missing expected raw file: {path}")
    datasets[name] = pd.read_csv(path)

print("Loaded datasets:")
for name, df in datasets.items():
    print(f"  {name:<25} {df.shape[0]:>6} rows x {df.shape[1]:>2} cols")

Loaded datasets:
  fund_master                   40 rows x 15 cols
  nav_history                46000 rows x  3 cols
  scheme_performance            40 rows x 19 cols
  investor_transactions      32778 rows x 13 cols
  portfolio_holdings           322 rows x  8 cols
  benchmark_indices           8050 rows x  3 cols


## Schema validation
Basic sanity checks — required columns present, no fully-empty files, correct row counts (40 schemes expected in `fund_master`).

In [2]:
required_columns = {
    "fund_master": ["amfi_code", "scheme_name", "category", "risk_category"],
    "nav_history": ["amfi_code", "date", "nav"],
    "scheme_performance": ["amfi_code", "risk_grade", "sharpe_ratio", "aum_crore"],
    "investor_transactions": ["investor_id", "amfi_code", "transaction_date",
                               "transaction_type", "amount_inr"],
    "portfolio_holdings": ["amfi_code", "sector", "weight_pct"],
    "benchmark_indices": ["date"],
}

for name, cols in required_columns.items():
    df = datasets[name]
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{name} is missing required columns: {missing}"
    assert not df.empty, f"{name} loaded empty"

assert datasets["fund_master"].shape[0] == 40, "Expected exactly 40 schemes in fund_master"

print("All schema checks passed.")

All schema checks passed.


## Type coercion
Parse date columns and coerce numeric fields so downstream notebooks don't need to repeat this.

In [3]:
datasets["nav_history"]["date"] = pd.to_datetime(datasets["nav_history"]["date"])
datasets["investor_transactions"]["transaction_date"] = pd.to_datetime(
    datasets["investor_transactions"]["transaction_date"]
)
datasets["benchmark_indices"]["date"] = pd.to_datetime(datasets["benchmark_indices"]["date"])

datasets["nav_history"]["nav"] = pd.to_numeric(datasets["nav_history"]["nav"], errors="coerce")
datasets["investor_transactions"]["amount_inr"] = pd.to_numeric(
    datasets["investor_transactions"]["amount_inr"], errors="coerce"
)

print("Date/numeric columns coerced.")
print("\nNull check after coercion:")
for name, df in datasets.items():
    n_nulls = df.isnull().sum().sum()
    print(f"  {name:<25} {n_nulls} null cells")

Date/numeric columns coerced.

Null check after coercion:
  fund_master               0 null cells
  nav_history               0 null cells
  scheme_performance        0 null cells
  investor_transactions     0 null cells
  portfolio_holdings        0 null cells
  benchmark_indices         0 null cells


## Save a raw ingestion snapshot
This notebook only validates and type-casts — actual cleaning (deduplication, gap-filling, merges) happens in `02_data_cleaning.ipynb`. Nothing is written to `data/processed/` yet, to keep ingestion and cleaning cleanly separated.

In [4]:
print("Ingestion complete. Datasets ready in memory:")
for name in datasets:
    print(" -", name)

Ingestion complete. Datasets ready in memory:
 - fund_master
 - nav_history
 - scheme_performance
 - investor_transactions
 - portfolio_holdings
 - benchmark_indices
